In [ ]:
#!/usr/bin/env python3
import os
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from torchcam.methods import GradCAM
from torchvision.transforms.functional import to_pil_image

# ====================== CONFIG ======================
DATA_DIR = './Aerial_Landscapes'
BATCH_SIZE = 32
NUM_EPOCHS = 10
NUM_CLASSES = 15
IMAGE_SIZE = 224
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# =====================================================

# ============ 1. 数据增强与加载 ============
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("🚀 Loading dataset...")
full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=val_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 替换训练集的 transform（不影响验证集 transform）
train_dataset.dataset.transform = train_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
print(f"✅ Dataset ready: {len(full_dataset)} images")

# ============ 2. 模型准备 ============
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)
print("✅ EfficientNet-B0 model loaded.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ============ 3. 训练与验证 ============
def train(model, loader):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels

# ============ 4. 主训练循环 ============
if __name__ == "__main__":
    train_accs, val_accs = [], []

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train(model, train_loader)
        val_loss, val_acc, _, _ = evaluate(model, val_loader)

        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"\n📘 Epoch {epoch+1}/{NUM_EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f}, Acc: {val_acc:.4f}")
        print("-" * 40)

    # ============ 5. 最终评估 ============
    _, _, all_preds, all_labels = evaluate(model, val_loader)
    print("📊 Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=full_dataset.classes))

    # ============ 6. 混淆矩阵 ============
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=full_dataset.classes,
                yticklabels=full_dataset.classes, cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    print("📌 Confusion matrix saved to confusion_matrix.png")

    # ============ 7. Grad-CAM 可视化 ============
    gradcam = GradCAM(model, target_layer="features.7")
    model.eval()

    # 可视化一个验证集中前几张图片
    images, labels = next(iter(val_loader))
    images = images.to(DEVICE)
    outputs = model(images)
    _, preds = torch.max(outputs, 1)

    for i in range(3):
        img_tensor = images[i].unsqueeze(0)
        activation_map = gradcam(img_tensor)[0]
        heatmap = activation_map.cpu().detach().numpy()
        orig = to_pil_image(images[i].cpu())

        # 显示图像 + heatmap
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(orig)
        plt.title(f"True: {full_dataset.classes[labels[i]]}\nPred: {full_dataset.classes[preds[i]]}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(orig)
        plt.imshow(heatmap, cmap='jet', alpha=0.5)
        plt.title("Grad-CAM")
        plt.axis('off')
        plt.savefig(f"gradcam_{i}.png")
        print(f"🔥 Grad-CAM saved to gradcam_{i}.png")

    # 保存模型
    torch.save(model.state_dict(), 'efficientnet_b0_skyview.pth')
    print("💾 Model saved to efficientnet_b0_skyview.pth")

    # 准确率可视化
    plt.figure()
    plt.plot(train_accs, label='Train Acc')
    plt.plot(val_accs, label='Val Acc')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("EfficientNet-B0 Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig("efficientnet_accuracy.png")
    print("📈 Accuracy plot saved to efficientnet_accuracy.png")
